#  GAN

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы: 
* https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html
* https://www.kaggle.com/datasets/splcher/animefacedataset
* https://github.com/eriklindernoren/PyTorch-GAN

## Задачи для совместного разбора

1\. Обсудите основные шаги в обучении GAN.

## Задачи для самостоятельного решения

In [28]:
import os
from PIL import Image
from matplotlib.image import imread
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.utils.data import DataLoader, DistributedSampler,Dataset
from torchvision import transforms, models
import matplotlib.pyplot as plt
import numpy as np

In [3]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


PyTorch version: 2.5.1+rocm6.2
CUDA available: True
CUDA version: None
Number of GPUs: 2
GPU 0: AMD Radeon RX 6700S
GPU 1: AMD Radeon 680M


In [4]:
torch.cuda.set_device(0)
print(f"Using device: {torch.cuda.get_device_name(torch.cuda.current_device())}")

Using device: AMD Radeon RX 6700S


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [1]:
import kagglehub

path = kagglehub.dataset_download("splcher/animefacedataset")

print("Path to dataset files:", path)

/home/nsedoff/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 395M/395M [04:22<00:00, 1.58MB/s] 

Extracting files...


Path to dataset files: /home/nsedoff/.cache/kagglehub/datasets/splcher/animefacedataset/versions/3


In [23]:
data_path = './animefacedataset/images'

In [26]:
class AnimeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.image_paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir) if f.endswith(('jpg', 'png', 'jpeg'))]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = imread(img_path)  # Загрузка изображения
            if self.transform:
                image = self.transform(image)
            return image
        except Exception as e:
            print(f"Ошибка загрузки файла {img_path}: {e}")
            return torch.zeros((3, 128, 128))  # Возврат пустого изображения в случае ошибки


In [30]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # Приведение всех изображений к размеру 128x128
    transforms.ToTensor(),         # Преобразование в тензор
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Нормализация
])

anime_dataset = AnimeDataset(data_path, transform=transform)

# Создание загрузчика данных
data_loader = DataLoader(anime_dataset, batch_size=16, shuffle=True)

# Получение примеров изображенийS
data_iter = iter(data_loader)
images = next(data_iter)

Ошибка загрузки файла ./animefacedataset/images/3725_2002.jpg: Unexpected type <class 'numpy.ndarray'>
Ошибка загрузки файла ./animefacedataset/images/2748_2002.jpg: Unexpected type <class 'numpy.ndarray'>
Ошибка загрузки файла ./animefacedataset/images/26632_2009.jpg: cannot identify image file '/home/nsedoff/PycharmProjects/ITiABD-PM22-7-Nikolay-Sedov/DL/animefacedataset/images/26632_2009.jpg'
Ошибка загрузки файла ./animefacedataset/images/22792_2008.jpg: Unexpected type <class 'numpy.ndarray'>
Ошибка загрузки файла ./animefacedataset/images/32534_2011.jpg: cannot identify image file '/home/nsedoff/PycharmProjects/ITiABD-PM22-7-Nikolay-Sedov/DL/animefacedataset/images/32534_2011.jpg'
Ошибка загрузки файла ./animefacedataset/images/21477_2008.jpg: Unexpected type <class 'numpy.ndarray'>
Ошибка загрузки файла ./animefacedataset/images/29543_2010.jpg: cannot identify image file '/home/nsedoff/PycharmProjects/ITiABD-PM22-7-Nikolay-Sedov/DL/animefacedataset/images/29543_2010.jpg'
Ошибка 

In [13]:
def imshow(img):
    img = img / 2 + 0.5  # Де-нормализация для отображения
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')

In [20]:
data_iter = iter(data_loader)
images = next(data_iter)

plt.figure(figsize=(10, 10))
imshow(torchvision.utils.make_grid(images))
plt.show()

UnidentifiedImageError: cannot identify image file '/home/nsedoff/PycharmProjects/ITiABD-PM22-7-Nikolay-Sedov/DL/animefacedataset/images/32840_2011.jpg'

<p class="task" id="1"></p>

1\. Создайте набор данных на основе архива `anime.zip`. Используя преобразования `torchvision`, приведите изображения к одному размеру и нормализуйте их. Выведите на экран несколько примеров изображений. 

- [ ] Проверено на семинаре

<p class="task" id="2"></p>

2\. Реализуйте архитектуру `DCGAN` и обучите модель. Подберите гиперпараметры таким образом, чтобы получаемые изображения стали достаточного качественными (четкими и без существенных дефектов). Во время обучения сохраняйте примеры генерации изображений из случайного шума и сравните, как менялось качество получаемых изображений в процессе обучения.

- [ ] Проверено на семинаре

<p class="task" id="3"></p>

3\. Создайте наборы данных на основе архива `summer2winter_yosemite.zip`. Используя преобразования `torchvision`, приведите изображения к одному размеру и нормализуйте их. Выведите на экран несколько примеров изображений, расположив изображения из одной пары рядом по горизонтали. 

- [ ] Проверено на семинаре

<p class="task" id="4"></p>

4\. Реализуйте архитектуру `CycleGAN` и обучите модель. Подберите гиперпараметры таким образом, чтобы получаемые изображения стали достаточного качественными (четкими и без существенных дефектов). Во время обучения сохраняйте примеры преобразования (в обе стороны) и  сравните, как менялось качество получаемых изображений в процессе обучения.

- [ ] Проверено на семинаре

## Обратная связь
- [ ] Хочу получить обратную связь по решению